# Infrastruktur-Daten via Google Places Area Insights API

**Notebook 2 der Datenbeschaffungs-Pipeline** — Schritt 6

Dieses Notebook fragt für alle **542 Planungsräume (PLR)** Berlins die Anzahl von 7 Infrastruktur-Typen ab
und erstellt die Datei `lor_infrastruktur.csv`, die in Notebook 3 in den Masterdatensatz eingebunden wird.

**Methode:** Pro PLR wird das vollständige LOR-Polygon (CCW-Winding-Order) als `customArea` übergeben.
Diese Methode wurde nach Tests mit Kreis-Abfragen und simplifiziertem Polygon gewählt,
da sie die präzisesten Ergebnisse liefert (exakte Planungsraumgrenzen statt Näherungskreis).

| Kategorie   | includedTypes                        | Output-Spalte     |
|-------------|--------------------------------------|-------------------|
| ÖPNV        | transit_station                      | transit_count     |
| Bildung     | university                           | university_count  |
| Bildung     | school                               | school_count      |
| Nachtleben  | bar, night_club                      | nightlife_count   |
| Arbeit      | corporate_office                     | office_count (+)  |
| Arbeit      | government_office                    | office_count (+)  |
| Wettbewerb  | fast_food_restaurant                 | fastfood_count    |

**Calls:** 542 PLR × 7 Typen = 3.794 Calls  
**Kosten:** ~$0.005/Call = ~$19  
**Laufzeit:** Bei 0.3s Sleep = ~22 Minuten  
**Resume-fähig:** Ja — bereits verarbeitete PLR werden übersprungen

In [2]:
import json
import time
import os
import math
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("../../.env"))

# ============================================================
# API KEY (aus .env Datei)
# ============================================================
API_KEY = os.getenv("GOOGLE_CLOUD_API_KEY", "")

if not API_KEY:
    raise ValueError("GOOGLE_CLOUD_API_KEY nicht in .env gefunden!")

print(f"API Key geladen: {API_KEY[:8]}...")

API Key geladen: AIzaSyBy...


In [3]:
# ============================================================
# Konfiguration
# ============================================================

LOR_GEOJSON    = Path("lor_planungsraeume_2021.geojson")
OUTPUT_JSON    = Path("dataset_infrastruktur.json")             # Zwischenspeicher (Resume-fähig)
OUTPUT_CSV     = Path("lor_infrastruktur.csv")                  # Finales Ergebnis

RADIUS_METER   = 500     # Radius pro Zentroid
SLEEP_SECONDS  = 0.3     # Pause zwischen Calls (Rate Limiting)

API_URL = "https://areainsights.googleapis.com/v1:computeInsights"

# 7 Call-Typen laut Projektplan
CALL_TYPES = [
    {"key": "transit_count",     "types": ["transit_station"]},
    {"key": "university_count",  "types": ["university"]},
    {"key": "school_count",      "types": ["school"]},
    {"key": "nightlife_count",   "types": ["bar", "night_club"]},
    {"key": "office_count_corp", "types": ["corporate_office"]},
    {"key": "office_count_gov",  "types": ["government_office"]},
    {"key": "fastfood_count",    "types": ["fast_food_restaurant"]},
]

print(f"Konfiguration:")
print(f"  LOR GeoJSON:  {LOR_GEOJSON}")
print(f"  Radius:       {RADIUS_METER}m")
print(f"  Sleep:        {SLEEP_SECONDS}s")
print(f"  Call-Typen:   {len(CALL_TYPES)}")
print(f"  API URL:      {API_URL}")

Konfiguration:
  LOR GeoJSON:  lor_planungsraeume_2021.geojson
  Radius:       500m
  Sleep:        0.3s
  Call-Typen:   7
  API URL:      https://areainsights.googleapis.com/v1:computeInsights


In [36]:
import math

def utm33n_to_wgs84(easting, northing):
    """Konvertiert EPSG:25833 (UTM Zone 33N) nach WGS84 (lat, lng in Grad)."""
    a = 6378137.0
    f = 1 / 298.257223563
    e2 = 2*f - f**2
    e_prime2 = e2 / (1 - e2)
    k0 = 0.9996
    E0 = 500000.0
    lambda0 = math.radians(15.0)

    E = easting - E0
    N = northing

    M = N / k0
    mu = M / (a * (1 - e2/4 - 3*e2**2/64 - 5*e2**3/256))

    e1 = (1 - math.sqrt(1 - e2)) / (1 + math.sqrt(1 - e2))

    phi1 = (mu
            + (3*e1/2 - 27*e1**3/32) * math.sin(2*mu)
            + (21*e1**2/16 - 55*e1**4/32) * math.sin(4*mu)
            + (151*e1**3/96) * math.sin(6*mu)
            + (1097*e1**4/512) * math.sin(8*mu))

    N1 = a / math.sqrt(1 - e2 * math.sin(phi1)**2)
    T1 = math.tan(phi1)**2
    C1 = e_prime2 * math.cos(phi1)**2
    R1 = a * (1 - e2) / (1 - e2 * math.sin(phi1)**2)**1.5
    D = E / (N1 * k0)

    lat = phi1 - (N1 * math.tan(phi1) / R1) * (
        D**2/2
        - (5 + 3*T1 + 10*C1 - 4*C1**2 - 9*e_prime2) * D**4/24
        + (61 + 90*T1 + 298*C1 + 45*T1**2 - 252*e_prime2 - 3*C1**2) * D**6/720
    )
    lng = lambda0 + (
        D
        - (1 + 2*T1 + C1) * D**3/6
        + (5 - 2*C1 + 28*T1 - 3*C1**2 + 8*e_prime2 + 24*T1**2) * D**5/120
    ) / math.cos(phi1)

    return math.degrees(lat), math.degrees(lng)

# ============================================================
# LOR GeoJSON laden + Zentroide berechnen (EPSG:25833 -> WGS84)
# Ring wird fuer den Polygon-API-Call mitgespeichert.
# ============================================================

with open(LOR_GEOJSON, encoding="utf-8") as f:
    geojson = json.load(f)

planungsraeume = []
for feat in geojson["features"]:
    props = feat["properties"]
    # Alle 542 Features sind MultiPolygon mit genau 1 Teil
    ring = feat["geometry"]["coordinates"][0][0]

    xs = [c[0] for c in ring]
    ys = [c[1] for c in ring]
    clat, clng = utm33n_to_wgs84(sum(xs)/len(xs), sum(ys)/len(ys))

    planungsraeume.append({
        "plr_id":   props["PLR_ID"],
        "plr_name": props["PLR_NAME"],
        "bez":      props["BEZ"],
        "lat":      clat,
        "lng":      clng,
        "ring":     ring,   # EPSG:25833, wird im Hauptloop zu WGS84 konvertiert
    })

print(f"Planungsraeume geladen: {len(planungsraeume)}")
print(f"Beispiel: plr_id={planungsraeume[0]['plr_id']}, plr_name={planungsraeume[0]['plr_name']}, lat={planungsraeume[0]['lat']:.4f}, lng={planungsraeume[0]['lng']:.4f}, vertices={len(planungsraeume[0]['ring'])}")


Planungsraeume geladen: 542
Beispiel: plr_id=11501341, plr_name=Karlshorst Süd, lat=52.4796, lng=13.5187, vertices=313


In [ ]:
# ============================================================
# Hilfsfunktionen für Polygon-basierte API-Abfragen
# ============================================================

def signed_area(coords):
    """Shoelace-Formel: positiv = CCW (Counter-Clockwise), negativ = CW.
    Wird benötigt um die Winding-Order des Polygons zu bestimmen."""
    n = len(coords)
    area = 0.0
    for i in range(n):
        x1, y1 = coords[i][0], coords[i][1]
        x2, y2 = coords[(i + 1) % n][0], coords[(i + 1) % n][1]
        area += (x1 * y2 - x2 * y1)
    return area / 2.0


def ensure_ccw(coords):
    """Stellt Counter-Clockwise Winding-Order sicher.
    Die Google Area Insights API verlangt CCW-Polygone (RFC 7946).
    LOR-Daten kommen in CW-Order -> müssen umgekehrt werden."""
    if signed_area(coords) < 0:  # CW -> umkehren
        return list(reversed(coords))
    return coords  # bereits CCW


def ring_to_wgs84(coords):
    """Konvertiert einen Polygon-Ring von EPSG:25833 nach WGS84.
    Gibt eine Liste von {latitude, longitude} Dicts zurück (API-Format)."""
    result = []
    for c in coords:
        lat, lng = utm33n_to_wgs84(c[0], c[1])
        result.append({'latitude': lat, 'longitude': lng})
    return result


def fetch_count_polygon(wgs84_coords, included_types):
    """Fragt die Area Insights API mit einem exakten Polygon (customArea) ab.
    Gibt die Anzahl der Places des angegebenen Typs im Polygon zurück,
    oder einen Fehler-String bei HTTP-Fehlern."""
    payload = {
        'insights': ['INSIGHT_COUNT'],
        'filter': {
            'locationFilter': {
                'customArea': {
                    'polygon': {
                        'coordinates': wgs84_coords  # Liste von {latitude, longitude}
                    }
                }
            },
            'typeFilter': {'includedTypes': included_types}
        }
    }
    headers = {'Content-Type': 'application/json', 'X-Goog-Api-Key': API_KEY}
    try:
        resp = requests.post(API_URL, json=payload, headers=headers, timeout=15)
        resp.raise_for_status()
        return int(resp.json().get('count', 0))  # API gibt count als String/Int zurück
    except requests.exceptions.HTTPError as e:
        return 'HTTP ' + str(e.response.status_code) + ': ' + e.response.text[:200]
    except Exception as e:
        return 'Fehler: ' + str(e)


print('Hilfsfunktionen bereit: signed_area, ensure_ccw, ring_to_wgs84, fetch_count_polygon')


Hilfsfunktionen bereit: signed_area, ensure_ccw, ring_to_wgs84, fetch_count_polygon


## Hauptloop — 542 PLR × 7 Typen

Der folgende Loop ruft für jeden der 542 Planungsräume alle 7 Infrastruktur-Typen ab.

**Technische Details:**
- **3.794 API-Calls** gesamt (542 × 7)
- **Resume-fähig:** Das Zwischenergebnis wird nach jedem PLR als JSON gespeichert (`dataset_infrastruktur.json`).
  Falls der Loop unterbrochen wird, überspringt er bereits verarbeitete PLR beim nächsten Start.
- **Polygon-Methode:** Jedes PLR-Polygon wird von EPSG:25833 nach WGS84 konvertiert und in CCW-Order gebracht,
  bevor es als `customArea` an die API übergeben wird.
- **office_count** = `office_count_corp` + `office_count_gov` (wird im Export-Schritt zusammengeführt)

In [41]:
# ============================================================
# Hauptloop — alle 542 PLR x 7 Typen (Option 3: volles Polygon)
# Resume-faehig: bereits verarbeitete PLR werden uebersprungen.
# Zwischenergebnisse werden nach jedem PLR in OUTPUT_JSON gespeichert.
# ============================================================

results = {}
if OUTPUT_JSON.exists():
    with open(OUTPUT_JSON, encoding='utf-8') as f:
        results = json.load(f)
    print('Resume: ' + str(len(results)) + ' PLR bereits verarbeitet, wird fortgesetzt...')
else:
    print('Starte neu.')

total_plr = len(planungsraeume)
est_min = (total_plr - len(results)) * len(CALL_TYPES) * SLEEP_SECONDS / 60

print('Gesamt: ' + str(total_plr) + ' PLR x ' + str(len(CALL_TYPES)) + ' Typen = ' + str(total_plr * len(CALL_TYPES)) + ' Calls')
print('Noch offen: ' + str(total_plr - len(results)) + ' PLR')
print('Geschaetzte Restlaufzeit: ' + str(round(est_min)) + ' Minuten')
print('Methode: Volles PLR-Polygon (Option 3)')
print()

start_time = datetime.now()

for i, plr in enumerate(planungsraeume):
    plr_id = plr['plr_id']

    if plr_id in results:
        continue

    ring_ccw = ensure_ccw(plr['ring'])
    wgs84_coords = ring_to_wgs84(ring_ccw)

    row = {
        'plr_id':       plr_id,
        'plr_name':     plr['plr_name'],
        'bez':          plr['bez'],
        'centroid_lat': plr['lat'],
        'centroid_lng': plr['lng'],
    }

    print('[' + str(i+1) + '/' + str(total_plr) + '] ' + plr['plr_name'] + ' (' + plr_id + ') — ' + str(len(wgs84_coords)) + ' Vertices')

    for call in CALL_TYPES:
        count = fetch_count_polygon(wgs84_coords, call['types'])
        row[call['key']] = count
        time.sleep(SLEEP_SECONDS)

    results[plr_id] = row

    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=1)

elapsed = (datetime.now() - start_time).seconds // 60
print('Fertig! ' + str(len(results)) + ' PLR verarbeitet in ' + str(elapsed) + ' Minuten.')
print('Zwischenergebnis gespeichert: ' + str(OUTPUT_JSON))

Starte neu.
Gesamt: 542 PLR x 7 Typen = 3794 Calls
Noch offen: 542 PLR
Geschaetzte Restlaufzeit: 19 Minuten
Methode: Volles PLR-Polygon (Option 3)

[1/542] Karlshorst Süd (11501341) — 313 Vertices
[2/542] Tirschenreuther Ring Ost (07601340) — 337 Vertices
[3/542] Wismarplatz (02500831) — 115 Vertices
[4/542] Märkisches Zentrum (12601134) — 412 Vertices
[5/542] Horstwalder Straße (07601547) — 228 Vertices
[6/542] Gropiusstadt Mitte (08301036) — 359 Vertices
[7/542] Spandauer Straße (05200421) — 470 Vertices
[8/542] Isenburger Weg (05200418) — 229 Vertices
[9/542] Rummelsburg (11501238) — 336 Vertices
[10/542] Schöneberger Linse (07200412) — 156 Vertices
[11/542] Frobenstraße (07100103) — 261 Vertices
[12/542] Rosenthal (03400618) — 304 Vertices
[13/542] Boulevard Kastanienallee (10200526) — 198 Vertices
[14/542] Odenwaldstraße (07300516) — 289 Vertices
[15/542] Gewerbegebiet Bitterfelder Straße (10100205) — 359 Vertices
[16/542] Lausitzer Platz (02300315) — 257 Vertices
[17/542] Wittels

## Export — CSV aufbereiten

Die Rohergebnisse aus dem JSON werden aufbereitet und als `lor_infrastruktur.csv` exportiert:
- `office_count_corp` + `office_count_gov` werden zu einem gemeinsamen `office_count` summiert
- Fehlende Werte (bei API-Fehlern) werden mit 0 aufgefüllt
- Die finale Spaltenreihenfolge entspricht der Zieltabelle im Projektplan

In [4]:
# ============================================================
# Ergebnisse aufbereiten + CSV exportieren
# ============================================================

with open(OUTPUT_JSON, encoding="utf-8") as f:
    results = json.load(f)

df = pd.DataFrame(results.values())

# office_count = corporate + government (summiert)
df["office_count"] = (
    df["office_count_corp"].fillna(0) +
    df["office_count_gov"].fillna(0)
).astype(int)

# Endgültige Spaltenreihenfolge (passt zur Zieltabelle im Projektplan)
final_cols = [
    "plr_id", "plr_name", "bez", "centroid_lat", "centroid_lng",
    "transit_count",
    "university_count",
    "school_count",
    "nightlife_count",
    "office_count",
    "fastfood_count",
]
df = df[final_cols]

# Fehlende Werte (None = API-Fehler) mit 0 auffüllen und als int speichern
count_cols = [c for c in final_cols if c.endswith("_count")]
df[count_cols] = df[count_cols].fillna(0).astype(int)

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"Exportiert: {OUTPUT_CSV}")
print(f"Zeilen: {len(df)}")
print()
print(df.describe().round(1))
print()
print(df.head(10).to_string())

Exportiert: lor_infrastruktur.csv
Zeilen: 542

       centroid_lat  centroid_lng  transit_count  university_count  \
count         542.0         542.0          542.0             542.0   
mean           52.5          13.4           14.1               1.2   
std             0.1           0.1           10.3               5.0   
min            52.4          13.1            0.0               0.0   
25%            52.5          13.3            7.0               0.0   
50%            52.5          13.4           12.0               0.0   
75%            52.5          13.5           19.0               1.0   
max            52.6          13.7           75.0              66.0   

       school_count  nightlife_count  office_count  fastfood_count  
count         542.0            542.0         542.0           542.0  
mean            8.8              6.2          12.1             1.7  
std             6.0              9.2          17.2             2.6  
min             0.0              0.0          